In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Notebook").master("local[*]").getOrCreate()
print("Spark", spark.version)

tx = spark.read.csv("../../data/transactions.csv", header=True, inferSchema=True)
tx.show(5)

Spark 4.1.3
+---+--------------+----------+-------------------+-------+---------+--------+----------+----------+--------------+-------+--------+
| id|transaction_id|account_id|          timestamp| amount|direction|currency|  category|  merchant|       channel|country|  status|
+---+--------------+----------+-------------------+-------+---------+--------+----------+----------+--------------+-------+--------+
|  1|     T00000001|    A00077|2026-10-09 10:32:11|2981.47|    debit|     EUR|  shopping| Decathlon|carte_en_ligne|     RU|accepted|
|  2|     T00000002|    A00049|2026-10-09 10:33:08|  86.31|    debit|     EUR|restaurant|Sushi Shop|carte_en_ligne|     FR|accepted|
|  3|     T00000003|    A00077|2026-10-09 10:34:05|   49.5|    debit|     EUR| transport|     Vélib|         carte|     FR|accepted|
|  4|     T00000004|    A00006|2026-10-09 10:35:02|  10.73|    debit|     EUR| transport|      RATP|carte_en_ligne|     FR|accepted|
|  5|     T00000005|    A00054|2026-10-09 10:35:59| 247.5

In [12]:
from pyspark.sql import functions as F
tx.select("transaction_id", "amount", "category", "country").show(5)
tx.filter(F.col("amount") > 1000).orderBy(F.col("amount").desc()).show(5)
tx.filter(F.col("country") != "FR").count()


+--------------+-------+----------+-------+
|transaction_id| amount|  category|country|
+--------------+-------+----------+-------+
|     T00000001|2981.47|  shopping|     RU|
|     T00000002|  86.31|restaurant|     FR|
|     T00000003|   49.5| transport|     FR|
|     T00000004|  10.73| transport|     FR|
|     T00000005| 247.57|  shopping|     IT|
+--------------+-------+----------+-------+
only showing top 5 rows
+---+--------------+----------+-------------------+-------+---------+--------+--------+-----------+--------------+-------+--------+
| id|transaction_id|account_id|          timestamp| amount|direction|currency|category|   merchant|       channel|country|  status|
+---+--------------+----------+-------------------+-------+---------+--------+--------+-----------+--------------+-------+--------+
| 84|     T00000084|    A00015|2026-10-09 11:51:02|3479.92|    debit|     EUR|  voyage|      Accor|carte_en_ligne|     RU|accepted|
|  1|     T00000001|    A00077|2026-10-09 10:32:11|2

10

In [13]:
debits = tx.filter((F.col("direction") == "debit") & (F.col("status") == "accepted"))

In [14]:
tx = tx.withColumn("ts", F.to_timestamp("timestamp")).withColumn("etranger", F.col("country") != "FR")
tx.select("timestamp", "ts", "country", "etranger").show(5)


+-------------------+-------------------+-------+--------+
|          timestamp|                 ts|country|etranger|
+-------------------+-------------------+-------+--------+
|2026-10-09 10:32:11|2026-10-09 10:32:11|     RU|    true|
|2026-10-09 10:33:08|2026-10-09 10:33:08|     FR|   false|
|2026-10-09 10:34:05|2026-10-09 10:34:05|     FR|   false|
|2026-10-09 10:35:02|2026-10-09 10:35:02|     FR|   false|
|2026-10-09 10:35:59|2026-10-09 10:35:59|     IT|    true|
+-------------------+-------------------+-------+--------+
only showing top 5 rows


In [15]:
debits.groupBy("category").agg(F.count("*").alias("nb"), F.round(F.sum("amount"), 2).alias("total"), F.round(F.avg("amount"), 2).alias("moyenne")).orderBy(F.col("total").desc()).show()

+------------+---+-------+-------+
|    category| nb|  total|moyenne|
+------------+---+-------+-------+
|    shopping| 14|7004.64| 500.33|
|      voyage|  5| 6279.6|1255.92|
|alimentation| 22|1234.49|  56.11|
|   transport| 23|1024.29|  44.53|
|  restaurant| 16| 946.07|  59.13|
|       sante|  8| 885.61|  110.7|
|     loisirs| 10| 855.53|  85.55|
+------------+---+-------+-------+



In [19]:
accounts = spark.read.csv("../../data/accounts.csv", header=True, inferSchema=True)
customers = spark.read.csv("../../data/customers.csv", header=True, inferSchema=True)
debits.join(accounts, "account_id").join(customers, "customer_id").groupBy("segment").agg(F.round(F.sum("amount"), 2).alias("total")).show()


+--------+-------+
| segment|  total|
+--------+-------+
|standard|1466.15|
| premium|6824.86|
+--------+-------+



In [20]:
tx.createOrReplaceTempView("transactions")
spark.sql("SELECT country, COUNT(*) AS nb, ROUND(SUM(amount), 2) AS total FROM transactions WHERE country <> 'FR' GROUP BY country ORDER BY total DESC").show()


+-------+---+-------+
|country| nb|  total|
+-------+---+-------+
|     RU|  2|6461.39|
|     US|  1| 1677.8|
|     IT|  2| 432.66|
|     ES|  2| 215.03|
|     DE|  2| 110.15|
|     BE|  1|  86.59|
+-------+---+-------+



In [21]:
gros = tx.filter(F.col("amount") > 2000)
gros.count()


3